# Pairwise alignments

---
## Imports

In [2]:
import numpy as np

---
## Implement Smith-Waterman algorithm


In [3]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back
    
    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1
        seq2 (str): sequence 2
        i (int): current row number
        j (int): current column number
        
    Returns:
        score in position (i,j)    
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT
        
    Pseudocode:
        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end
        
    '''
    row_letter = seq1[i-1]  #Get this row's letter: seq1 at position (i-1)
    col_letter = seq2[j-1]  #Get this column's letter: seq2 at position (j-1)
    
    #Diagonal number:
    if row_letter == col_letter:    #If this row's letter equals this column's letter
        diag_score = matrix[i - 1, j - 1] + match  #diag_score = matrix at (i-1, j-1) + match
    else:        #Otherwise:
        diag_score = matrix[i - 1, j - 1] + mismatch  #diag_score = matrix at (i-1, j-1) + mismatch
    up_score = matrix[i - 1, j] + gap  #up_score = matrix at (i-1, j) + gap
    left_score = matrix[i, j-1] + gap  #left_score = matrix at (i, j-1) + gap

    score = max(0, diag_score, up_score, left_score)  #score = biggest of: 0, diag_score, up_score, left_score

    #Pick the move (check in this order, first match wins if tie score):
    if score == 0:   #IF score is 0:  move = 0 (END)
        move = 0
    elif score == diag_score:  #ELIF score = diag:  move = 1 (DIAG)
        move = 1
    elif score == up_score:  #ElIF score = up:  move = 2 (UP)
        move = 2
    else:    #ELSE:  move = 3 (LEFT)
        move = 3

    return score, move  #Return score and move


In [4]:
#Sanity check - test calc_score function
test_grid = np.zeros((4, 4))
test_grid[2, 1] = 1
print(cal_score(test_grid, "ACT", "CTT", 3, 2, 1, -1, -1))

(np.float64(2.0), 1)


In [5]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''Find the optimal path through scoring marix
        
        diagonal: match/mismatch
        up: gap in seq1
        left: gap in seq2
        
    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from
        
    Returns:
        aligned_seq1 (str): e.g. GTTGAC
        aligned_seq2 (str): e.g. GTT-AC
        
    Pseudocode:
        while current_move != END:
            current_move = traceback_matrix[current_row][current_col]
            if current_move == DIAG:
                ...
            elif current_move == UP:
                ...
            elif current_move == LEFT:
                ...
            
    '''
    aligned_seq1 = ""  #aligned_seq1 = empty string
    aligned_seq2 = ""  #aligned_seq2 = empty string
    current_row, current_col = maximum_position  #current_row, current_col = maximum_position (row, column)
    current_move = traceback_matrix[current_row, current_col]  #current_move = traceback_matrix at (current_row, current_col)
    
    while current_move != 0:  #WHILE current_move is not 0 (END):
        if current_move == 1:  #IF current_move is 1 (DIAG):
            aligned_seq1 = seq1[current_row - 1] + aligned_seq1   #add seq1 letter at (current_row - 1) to front of aligned_seq1
            aligned_seq2 = seq2[current_col - 1] + aligned_seq2   #add seq2 letter at (current_col - 1) to front of aligned_seq2
            current_row = current_row - 1    #move up one row and left one column
            current_col = current_col -1
    
        elif current_move == 2:  #ELIF current_move is 2 (UP):
            aligned_seq1 = seq1[current_row - 1] + aligned_seq1   #add seq1 letter at (currnet_row - 1) to front of aligned_seq1
            aligned_seq2 = "-" + aligned_seq2   #add "-" to front of aligned_seq2
            current_row = current_row - 1   #move up one row
    
        elif current_move == 3:   #ELIF current_move is 3 (LEFT): 
            aligned_seq1 = "-" + aligned_seq1   #add "-" to front of aligned_seq1
            aligned_seq2 = seq2[current_col - 1] + aligned_seq2    #add seq2 letter at (current_col - 1) to front of aligned_seq2
            current_col = current_col - 1   #move left one column
    
        current_move = traceback_matrix[current_row, current_col]   #current_move = traceback_matrix at (current_row, current_col)
    
    return aligned_seq1, aligned_seq2    #Return aligned_seq1, aligned_seq2

            

In [6]:
#Sanity check - test traceback function
test_arrows = np.array([[0, 0, 0, 0],
                        [0, 0, 0, 0],
                        [0, 1, 0, 0],
                        [0, 0, 1, 1]])
print(traceback("ACT", "CTT", test_arrows, (3, 2)))

('CT', 'CT')


In [7]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1):
    '''Smith-Waterman algorithm for local alignment
    
    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +1
        mismatch: default = -1
        gap: default = -1
    
    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix
    '''
    rows = len(seq1) + 1  #rows = length of seq1 + 1
    cols = len(seq2) + 1  #cols = length of seq2 + 1
    score_matrix = np.zeros((rows, cols))  #score_matrix = grid of zeros, size rows x cols (decimals, not whole number, so that gaps like -0.5 work)
    traceback_matrix = np.zeros((rows, cols), dtype=int)   #traceback_matrix = grid of zeros, size rows x cols (whole numbers 0, 1, 2, 3)
    
    for i in range(1, rows):    #for i from 1 to rows -1:
        for j in range(1, cols):    #for j from 1 to cols -1:
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)  #score, move = cal_score (score_matrix, seq1, seq2, i, j, match, mismatch, gap)
            score_matrix[i, j] = score   #score_matrix at (i, j) = score
            traceback_matrix[i, j] = move   #traceback_matrix at (i, j) = move
    
    best_score = np.max(score_matrix)    #best_score = highest number in score matrix
    max_pos = np.unravel_index(np.argmax(score_matrix), score_matrix.shape)    #max_pos = (row, column) where best_score is (if there is a tie, this keeps only the first one found)

    if best_score == 0:  #if best_score is 0 (END) display that and makes sure it doesn't get caught up in loop
        print("No local alignment found (best score is 0)")

    print("Alignment score: ", best_score)  #print best_score
    
    aligned_seq1, aligned_seq2 = traceback(seq1, seq2, traceback_matrix, max_pos)   #aligned_seq1, aligned_seq2 = traceback (seq1, seq2, traceback_matrix, max_pos)
    
    return aligned_seq1, aligned_seq2, score_matrix   #Return aligned_seq1, aligned_seq2, score_matrix

In [8]:
# Example from slides
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1, seq2)

print (aligned_seq1)
print (aligned_seq2)
print (score_matrix)

Alignment score:  4.0
AC-TTA
ACATTA
[[0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0. 0. 2. 1.]
 [0. 1. 0. 2. 1. 0. 0. 1. 1.]
 [0. 0. 0. 1. 1. 2. 1. 0. 0.]
 [0. 0. 0. 0. 0. 2. 3. 2. 1.]
 [0. 0. 1. 0. 1. 1. 2. 4. 3.]
 [0. 0. 0. 0. 0. 0. 1. 3. 3.]]


In [9]:
seq3 = "GGGGAAAAAAAAAAAACCCC"
seq4 = "GGGGAAAAAACCCC"

aligned_seq3, aligned_seq4, score_matrix = smith_waterman(seq3,seq4)

print (aligned_seq3)
print (aligned_seq4)
print (score_matrix)

Alignment score:  10.0
GGGGAAAAAA
GGGGAAAAAA
[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  1.  1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  2.  2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  3.  3.  2.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  2.  3.  4.  3.  2.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  2.  3.  5.  4.  3.  2.  1.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  2.  4.  6.  5.  4.  3.  2.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  1.  3.  5.  7.  6.  5.  4.  3.  2.  1.  0.]
 [ 0.  0.  0.  0.  0.  2.  4.  6.  8.  7.  6.  5.  4.  3.  2.]
 [ 0.  0.  0.  0.  0.  1.  3.  5.  7.  9.  8.  7.  6.  5.  4.]
 [ 0.  0.  0.  0.  0.  1.  2.  4.  6.  8. 10.  9.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  5.  7.  9.  9.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  6.  8.  8.  8.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  5.  7.  7.  7.  7.  6.]
 [ 0.  0.  0.  0.  0.  1.  2.  3.  4.  5.  6.  6.  6.  6.  6.]
 [ 0.  0. 

In [10]:
#Extra Test:  show the gap penalty point working
aligned_seq3, aligned_seq4, score_matrix = smith_waterman(seq3, seq4, gap=-0.5)
print(aligned_seq3)
print(aligned_seq4)

Alignment score:  11.0
GGGGAAAAAAAAAAAACCCC
GGGG------AAAAAACCCC
